In [1]:
# ============================================================
# Produce latency + mAP results table
# ============================================================
#
# What this notebook does:
# 1) Runs bench.py for torch / ort_fp32 / ort_int8 in forward and e2e modes
# 2) Runs eval.py for torch / ort_fp32 / ort_int8 to get mAP@0.5
# 3) Builds a tidy results table + deltas
# ============================================================

import os
import re
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

# ----------------------------
# USER CONFIG
# ----------------------------
REPO_ROOT = Path.home() / "repos" / "automotive-ssd-object-detection" # Path(r"C:\Users\eblac\Documents\GitHub\self-driving-car")
WEIGHTS = REPO_ROOT / "v2" / "saved_models" / "DIoU_mAP_551_iou_thresh_45_max_img_per_det_200.pth" # Path(r"C:\Users\eblac\Documents\GitHub\self-driving-car\app_files\saved_models\noZoomOut_Bootstrap.pth")
TRAIN_DIR = Path.home() / "datasets" / "Udacity_car_data" / "data" / "train" # Path(r"C:\Udacity_car_data\data\train")
ONNX_FP32 = REPO_ROOT / "PTQ_testing" / "ssd_v2.onnx"
ONNX_INT8 = REPO_ROOT / "PTQ_testing" / "ssd_int8_v2.onnx"

THREADS = 4
BATCH_LAT = 1
RUNS = 200
WARMUP = 20

# mAP evaluation batch size (separate from latency batch)
BATCH_EVAL = 8

# postprocess settings (keep fixed across backends)
SCORE_THRESH = 0.2
NMS_THRESH = 0.5
MAX_PER_IMG = 100
CLASS_AGNOSTIC = False

# Where your scripts live
SCRIPT_DIR = REPO_ROOT / "PTQ_testing"
BENCH_PY = SCRIPT_DIR / "bench.py"
EVAL_PY = SCRIPT_DIR / "eval.py"

assert REPO_ROOT.exists(), REPO_ROOT
assert WEIGHTS.exists(), WEIGHTS
assert ONNX_FP32.exists(), ONNX_FP32
assert ONNX_INT8.exists(), ONNX_INT8
assert BENCH_PY.exists(), BENCH_PY
assert EVAL_PY.exists(), EVAL_PY

# Ensure subprocess calls use repo root
def run_cmd(cmd, cwd=REPO_ROOT):
    """Run a command and return stdout+stderr combined."""
    proc = subprocess.run(
        cmd,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        shell=False,
    )
    out = proc.stdout + "\n" + proc.stderr
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}):\n{cmd}\n\nOutput:\n{out}")
    return out

# Parse "p50=..., p95=..., mean=..." line from bench.py output
BENCH_RE = re.compile(r"p50=([\d\.]+)\s*ms,\s*p95=([\d\.]+)\s*ms,\s*mean=([\d\.]+)\s*ms", re.IGNORECASE)

def parse_bench_output(output: str):
    m = BENCH_RE.search(output)
    if not m:
        raise ValueError(f"Could not parse bench output:\n{output}")
    return {"p50_ms": float(m.group(1)), "p95_ms": float(m.group(2)), "mean_ms": float(m.group(3))}

# Parse "backend: mAP@0.5 = X.XXXX" lines from eval.py output
EVAL_RE = re.compile(r"^(torch|ort_fp32|ort_int8):\s*mAP@0\.5\s*=\s*([0-9]*\.?[0-9]+)\s*$", re.MULTILINE)

def parse_eval_output(output: str):
    found = dict(EVAL_RE.findall(output))
    if not found:
        raise ValueError(f"Could not parse eval output:\n{output}")
    return {k: float(v) for k, v in found.items()}

# ----------------------------
# 1) Run latency benchmarks
# ----------------------------
backends = ["torch", "ort_fp32", "ort_int8"]
modes = ["forward", "e2e"]

latency_rows = []
for backend in backends:
    for mode in modes:
        cmd = [
            sys.executable, str(BENCH_PY),
            "--backend", backend,
            "--mode", mode,
            "--batch", str(BATCH_LAT),
            "--runs", str(RUNS),
            "--warmup", str(WARMUP),
            "--threads", str(THREADS),
            "--weights", str(WEIGHTS),
            "--score_thresh", str(SCORE_THRESH),
            "--nms_thresh", str(NMS_THRESH),
            "--max_per_img", str(MAX_PER_IMG),
        ]
        if CLASS_AGNOSTIC:
            cmd.append("--class_agnostic")

        if backend == "ort_fp32":
            cmd += ["--onnx_fp32", str(ONNX_FP32)]
        if backend == "ort_int8":
            cmd += ["--onnx_int8", str(ONNX_INT8)]

        out = run_cmd(cmd)
        stats = parse_bench_output(out)
        latency_rows.append({"backend": backend, "mode": mode, **stats})

lat_df = pd.DataFrame(latency_rows)
lat_df

# ----------------------------
# 2) Run accuracy evaluation (mAP@0.5)
# ----------------------------
eval_cmd = [
    sys.executable, str(EVAL_PY),
    "--weights", str(WEIGHTS),
    "--train_path", str(TRAIN_DIR),
    "--onnx_fp32", str(ONNX_FP32),
    "--onnx_int8", str(ONNX_INT8),
    "--batch", str(BATCH_EVAL),
    "--threads", str(THREADS),
    "--score_thresh", str(SCORE_THRESH),
    "--nms_thresh", str(NMS_THRESH),
    "--max_per_img", str(MAX_PER_IMG),
    "--backends", "torch", "ort_fp32", "ort_int8",
]
if CLASS_AGNOSTIC:
    eval_cmd.append("--class_agnostic")

eval_out = run_cmd(eval_cmd)
map_dict = parse_eval_output(eval_out)

map_df = pd.DataFrame([{"backend": k, "map_50": v} for k, v in map_dict.items()])
map_df

# ----------------------------
# 3) Build results table
# ----------------------------
# Choose which latency mode to report in README:
# "e2e" or "forward"
REPORT_MODE = "e2e"

lat_report = lat_df[lat_df["mode"] == REPORT_MODE][["backend", "p50_ms", "p95_ms"]].copy()
res = lat_report.merge(map_df, on="backend", how="left")

# Add deltas vs torch
torch_row = res[res["backend"] == "torch"].iloc[0]
res["delta_p50_ms_vs_torch"] = res["p50_ms"] - torch_row["p50_ms"]
res["delta_p95_ms_vs_torch"] = res["p95_ms"] - torch_row["p95_ms"]
res["delta_map50_vs_torch"] = res["map_50"] - torch_row["map_50"]

# Nice ordering and labels
backend_order = {"torch": 0, "ort_fp32": 1, "ort_int8": 2}
res["order"] = res["backend"].map(backend_order)
res = res.sort_values("order").drop(columns=["order"])

res

# ----------------------------
# 4) Emit a Markdown table you can paste into README
# ----------------------------
def fmt(x, nd=3):
    if pd.isna(x):
        return ""
    return f"{x:.{nd}f}"

md = []
md.append("| Backend | p50 latency (ms) | p95 latency (ms) | mAP@0.5 | Δp50 vs torch (ms) | ΔmAP@0.5 vs torch |")
md.append("|---|---:|---:|---:|---:|---:|")
for _, r in res.iterrows():
    md.append(
        "| {backend} | {p50} | {p95} | {map50} | {dp50} | {dmap} |".format(
            backend=r["backend"],
            p50=fmt(r["p50_ms"], 3),
            p95=fmt(r["p95_ms"], 3),
            map50=fmt(r["map_50"], 4),
            dp50=fmt(r["delta_p50_ms_vs_torch"], 3),
            dmap=fmt(r["delta_map50_vs_torch"], 4),
        )
    )

print("\n".join(md))

# Optional: save the table to a file
out_path = REPO_ROOT / "ptq_results_table_v2.md"
out_path.write_text("\n".join(md), encoding="utf-8")
print(f"\nWrote: {out_path}")

| Backend | p50 latency (ms) | p95 latency (ms) | mAP@0.5 | Δp50 vs torch (ms) | ΔmAP@0.5 vs torch |
|---|---:|---:|---:|---:|---:|
| torch | 226.004 | 291.396 | 0.5544 | 0.000 | 0.0000 |
| ort_fp32 | 180.391 | 228.063 | 0.5544 | -45.613 | 0.0000 |
| ort_int8 | 44.702 | 55.280 | 0.5540 | -181.302 | -0.0004 |

Wrote: /home/eblackstone/repos/automotive-ssd-object-detection/ptq_results_table_v2.md
